### Stabilumo analizė   

In [1]:
import pandas as pd
import numpy as np

scaled_features = pd.read_excel("Outputs/scaled_features.xlsx")

scaled_features

,account_number,mean_call_duration,max_call_duration,unique_contacts_called,unique_destination_codes,unique_numbers_per_account,unique_active_days,days_between_first_and_last_call,active_day_ratio,avg_calls_per_active_day,avg_calls_per_own_number,night_calls_count,day_calls_count,max_calls_to_single_contact,mean_calls_to_single_contact,answered_calls_count,busy_calls_count,no_answer_calls_count,cancelled_calls_count,webrtc_calls_count
0,0000df940049fa1bbefc699a17671295,0.189056,0.418073,1.588235,2.125,0,0.655172,-0.129611,0.463529,0.130939,0.755952,0.000000,0.723032,1.106667,-0.097895,0.939850,0.000,0.000000,0.157895,0
1,000a38dacb7099ae710216b9cd54087c,2.795097,1.116928,-0.117647,0.000,0,0.494253,0.237288,0.014483,-0.599371,0.250000,0.000000,0.227405,0.533333,0.704444,0.315789,0.250,-0.181818,-0.052632,0
2,000b1bddfe23f65af9990f52b3ae004e,-0.201718,-0.491371,-0.411765,-0.375,2,-0.459770,-0.911266,-0.157743,1.152263,-0.458333,0.000000,-0.443149,-0.400000,-0.490000,-0.406015,-0.250,-0.181818,-0.421053,0
3,000d7412614cfc8e6d918ca63e1f37dc,0.623868,0.175296,-0.294118,-0.250,0,-0.436782,-0.556331,-0.546958,2.263374,-0.375000,0.000000,-0.384840,-0.360000,-0.373333,-0.345865,0.000,-0.181818,-0.368421,0
4,0013a4303bdffc2658cd40f80227a524,-0.207689,0.331976,1.705882,1.000,0,3.563218,0.436690,1.680298,1.104915,5.726190,44.666667,4.810496,9.413333,1.962500,4.932331,12.375,2.545455,9.631579,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9598,fff3d1a7b92ca72dbca37b0ba062af95,1.112639,0.867171,-0.529412,-0.625,0,0.126437,0.227318,-0.228444,-0.524885,0.000000,0.000000,-0.017493,0.453333,3.310000,0.030075,-0.250,0.909091,-0.315789,0
9599,fff65074634a70efc96bc738e7bf368c,0.613524,-0.165212,0.294118,0.125,0,0.218391,0.061815,-0.088785,-0.596371,0.047619,0.000000,0.029155,0.066667,-0.096250,0.127820,-0.250,-0.181818,-0.157895,0
9600,fff8217ecefc2d900e361d22d6e90ce0,0.621712,1.116928,2.058824,1.750,0,2.367816,0.492522,0.931453,1.049217,3.821429,5.333333,3.632653,6.746667,0.927391,3.894737,9.125,0.181818,2.526316,0
9601,fff86cfbdc7d0f8b59b65db318798076,-0.639242,-0.236572,-0.117647,0.375,0,0.908046,-0.253240,0.948356,-0.454716,0.648810,0.333333,0.612245,1.413333,1.448889,0.729323,-0.250,-0.181818,0.578947,0


In [2]:
# Bazinis X jau turi būti normalizuotas (scaled_features be account_number)
account_numbers = scaled_features["account_number"]
X = scaled_features.drop(columns=["account_number"])

# --- Jaccard funkcija ---
def jaccard(set_a, set_b):
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return round(inter / union, 3) if union > 0 else 0


In [3]:
#  iForest stabilumo analizė
from sklearn.ensemble import IsolationForest

# --- iForest su 5 contamination reikšmėmis ---
contamination_values = [0.005, 0.01, 0.015, 0.02, 0.025]
if_contamination_sets = {}

for c in contamination_values:
    model = IsolationForest(
        n_estimators=200,
        contamination=c,
        max_samples="auto",
        random_state=42
    )
    preds = model.fit_predict(X)
    anomaly_ids = set(account_numbers[preds == -1])
    if_contamination_sets[c] = anomaly_ids
    print(f"iForest contamination={c}: {len(anomaly_ids)} vartotojų")

# --- iForest su 5 n_estimators reikšmėmis ---
n_estimators_values = [100, 150, 200, 250, 300]
if_tree_sets = {}

for n in n_estimators_values:
    model = IsolationForest(
        n_estimators=n,
        contamination=0.01,
        max_samples="auto",
        random_state=42
    )
    preds = model.fit_predict(X)
    anomaly_ids = set(account_numbers[preds == -1])
    if_tree_sets[n] = anomaly_ids
    print(f"iForest n_estimators={n}: {len(anomaly_ids)} vartotojų")


# --- iForest Jaccard palyginimas contamination ---
if_contamination_comparisons = [
    (0.01, 0.005),
    (0.01,  0.015),
    (0.01, 0.02),
    (0.01, 0.025),
]

# --- iForest Jaccard palyginimas medziams ---
if_tree_comparisons = [
    (200, 100),
    (200, 150),
    (200, 250),
    (200, 300),
]

print("\n--- iForest stabilumo analizė pagal contamination ---")
if_results_contamination = []
for p1, p2 in if_contamination_comparisons:
    j = jaccard(if_contamination_sets[p1], if_contamination_sets[p2])
    same_users = len(if_contamination_sets[p1] & if_contamination_sets[p2])
    print(f"cont={p1} vs cont={p2}: J={j}  (sutampa: {same_users})")
    if_results_contamination.append({"Metodas": "iForest", "Palyginimas": f"contamination={p1} vs contamination={p2}",
                        "Sutampančių": same_users, "Jaccard": j})
    
print("\n--- iForest papildoma stabilumo analizė pagal n_estimators ---")
if_results_tree = []
for p1, p2 in if_tree_comparisons:
    j = jaccard(if_tree_sets[p1], if_tree_sets[p2])
    same_users = len(if_tree_sets[p1] & if_tree_sets[p2])
    print(f"n_est={p1} vs n_est={p2}: J={j}  (sutampa: {same_users})")
    if_results_tree.append({"Metodas": "iForest", "Palyginimas": f"n_estimators={p1} vs n_estimators={p2}",
                        "Sutampančių": same_users, "Jaccard": j})

iForest contamination=0.005: 49 vartotojų
iForest contamination=0.01: 97 vartotojų
iForest contamination=0.015: 145 vartotojų
iForest contamination=0.02: 193 vartotojų
iForest contamination=0.025: 241 vartotojų
iForest n_estimators=100: 97 vartotojų
iForest n_estimators=150: 97 vartotojų
iForest n_estimators=200: 97 vartotojų
iForest n_estimators=250: 97 vartotojų
iForest n_estimators=300: 97 vartotojų

--- iForest stabilumo analizė pagal contamination ---
cont=0.01 vs cont=0.005: J=0.505  (sutampa: 49)
cont=0.01 vs cont=0.015: J=0.669  (sutampa: 97)
cont=0.01 vs cont=0.02: J=0.503  (sutampa: 97)
cont=0.01 vs cont=0.025: J=0.402  (sutampa: 97)

--- iForest papildoma stabilumo analizė pagal n_estimators ---
n_est=200 vs n_est=100: J=0.98  (sutampa: 96)
n_est=200 vs n_est=150: J=0.98  (sutampa: 96)
n_est=200 vs n_est=250: J=0.98  (sutampa: 96)
n_est=200 vs n_est=300: J=0.96  (sutampa: 95)


In [4]:
# LOF stabilumo analizė
import warnings
from sklearn.neighbors import LocalOutlierFactor
warnings.filterwarnings('ignore')

# --- LOF su 5 n_neighbors reikšmėmis ---
lof_neighbors_params = [10, 15, 20, 25, 30]
lof_neighbors_sets = {}
    
for k in lof_neighbors_params:
    model = LocalOutlierFactor(
        n_neighbors=k,
        contamination=0.01,
        metric="euclidean"
    )
    preds = model.fit_predict(X)
    anomaly_ids = set(account_numbers[preds == -1])
    lof_neighbors_sets[k] = anomaly_ids
    print(f"LOF k={k}: {len(anomaly_ids)} vartotojų")

# --- LOF su 5 contamination reikšmėmis ---
lof_contamination_params = [0.005, 0.01, 0.015, 0.02, 0.025]
lof_contamination_sets = {}

for n in lof_contamination_params:
    model = LocalOutlierFactor(
        n_neighbors=20,
        contamination=n,
        metric="euclidean"
    )
    preds = model.fit_predict(X)
    anomaly_ids = set(account_numbers[preds == -1])
    lof_contamination_sets[n] = anomaly_ids
    print(f"LOF contamination={n}: {len(anomaly_ids)} vartotojų")

# --- LOF Jaccard kaimynų palyginimas ---
lof_neighbors_comparisons = [
    (20, 10),
    (20, 15),
    (20, 25),
    (20, 30),
    ]

# --- LOF Jaccard contamination palyginimas ---
lof_contamination_comparisons = [
    (0.01, 0.005),
    (0.01, 0.015),
    (0.01, 0.02),
    (0.01, 0.025),
]


print("\n--- LOF stabilumo analizė pagal kaimynus ---")
lof_results_neighbors = []
for p1, p2 in lof_neighbors_comparisons:
    j = jaccard(lof_neighbors_sets[p1], lof_neighbors_sets[p2])
    inter = len(lof_neighbors_sets[p1] & lof_neighbors_sets[p2])
    print(f"k={p1} vs k={p2}: J={j}  (sutampa: {inter})")
    lof_results_neighbors.append({"Metodas": "LOF", "Palyginimas": f"k={p1} vs k={p2}",
                        "Sutampančių": inter, "Jaccard": j})
    
print("\n--- LOF stabilumo analizė pagal contamination ---")
lof_results_contamination = []
for p1, p2 in lof_contamination_comparisons:
    j = jaccard(lof_contamination_sets[p1], lof_contamination_sets[p2])
    inter = len(lof_contamination_sets[p1] & lof_contamination_sets[p2])
    print(f"cont={p1} vs cont={p2}: J={j}  (sutampa: {inter})")
    lof_results_contamination.append({"Metodas": "LOF", "Palyginimas": f"contamination={p1} vs contamination={p2}",
                        "Sutampančių": inter, "Jaccard": j})


LOF k=10: 97 vartotojų
LOF k=15: 97 vartotojų
LOF k=20: 97 vartotojų
LOF k=25: 97 vartotojų
LOF k=30: 97 vartotojų
LOF contamination=0.005: 49 vartotojų
LOF contamination=0.01: 97 vartotojų
LOF contamination=0.015: 145 vartotojų
LOF contamination=0.02: 193 vartotojų
LOF contamination=0.025: 241 vartotojų

--- LOF stabilumo analizė pagal kaimynus ---
k=20 vs k=10: J=0.47  (sutampa: 62)
k=20 vs k=15: J=0.702  (sutampa: 80)
k=20 vs k=25: J=0.796  (sutampa: 86)
k=20 vs k=30: J=0.617  (sutampa: 74)

--- LOF stabilumo analizė pagal contamination ---
cont=0.01 vs cont=0.005: J=0.505  (sutampa: 49)
cont=0.01 vs cont=0.015: J=0.669  (sutampa: 97)
cont=0.01 vs cont=0.02: J=0.503  (sutampa: 97)
cont=0.01 vs cont=0.025: J=0.402  (sutampa: 97)


In [5]:
#  Suvestinė lentelė
# --- Bendra suvestinė lentelė ---
stability_table = pd.DataFrame(if_results_contamination + if_results_tree + lof_results_neighbors + lof_results_contamination)
print("\n=== Stabilumo analizės suvestinė ===")
print(stability_table.to_string(index=False))

# Pasirinktinai: eksportas
# stability_table.to_excel("stabilumo_analize.xlsx", index=False)



=== Stabilumo analizės suvestinė ===
Metodas                               Palyginimas  Sutampančių  Jaccard
iForest contamination=0.01 vs contamination=0.005           49    0.505
iForest contamination=0.01 vs contamination=0.015           97    0.669
iForest  contamination=0.01 vs contamination=0.02           97    0.503
iForest contamination=0.01 vs contamination=0.025           97    0.402
iForest      n_estimators=200 vs n_estimators=100           96    0.980
iForest      n_estimators=200 vs n_estimators=150           96    0.980
iForest      n_estimators=200 vs n_estimators=250           96    0.980
iForest      n_estimators=200 vs n_estimators=300           95    0.960
    LOF                              k=20 vs k=10           62    0.470
    LOF                              k=20 vs k=15           80    0.702
    LOF                              k=20 vs k=25           86    0.796
    LOF                              k=20 vs k=30           74    0.617
    LOF contamination=0.01